In [9]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [10]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [11]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "chicago_taxi_trips" dataset
dataset_ref = client.dataset("chicago_taxi_trips", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "taxi_trips" table
table_ref = dataset_ref.table("taxi_trips")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe(create_bqstorage_client=False)

,unique_key,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_latitude,pickup_longitude,pickup_location,dropoff_latitude,dropoff_longitude,dropoff_location
0,b99416692f5bc01cdb444cbcec7e1105d30d455d,8dd990653cc2793d96b80ae45e928afc9a590b21b491ee...,2019-05-29 18:00:00+00:00,2019-05-29 18:30:00+00:00,1500,1.50,<NA>,<NA>,<NA>,<NA>,...,0.0,12.36,Credit Card,KOAM Taxi Association,NaN,NaN,NaN,NaN,NaN,NaN
1,b99887012c8beccc1dae3f6fcde2fc94ff0ef7b1,d156d4adff01addd62b65753ce67dc8e04bdaf60f7eb14...,2019-05-20 19:15:00+00:00,2019-05-20 19:30:00+00:00,506,1.67,<NA>,<NA>,<NA>,<NA>,...,0.0,10.50,Credit Card,Chicago Carriage Cab Corp,NaN,NaN,NaN,NaN,NaN,NaN
2,b98b6b7afe1f8065d8ba71cd023d9a13f2460f84,beefd3462e3f5a8e854942a2796876f6db73ebbd25b435...,2019-05-08 00:00:00+00:00,2019-05-08 00:00:00+00:00,60,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,3.25,Cash,Taxi Affiliation Services,NaN,NaN,NaN,NaN,NaN,NaN
3,b9865af3673441b1a79973ad08bd9909bd422af4,ef249ea2c31ccb20ce9eb258d3116254a6f9fe05c7cc84...,2019-05-07 16:30:00+00:00,2019-05-07 17:00:00+00:00,1122,2.86,<NA>,<NA>,<NA>,<NA>,...,1.0,13.25,Cash,Taxi Affiliation Service Yellow,NaN,NaN,NaN,NaN,NaN,NaN
4,e04dbd9a8ba723cc785c4b0398106b72e16d7870,45002471b85d61e62cf602a15145d2bff912e080f50470...,2013-05-16 08:15:00+00:00,2013-05-16 08:15:00+00:00,<NA>,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,30.06,Credit Card,Chicago Elite Cab Corp.,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Fill in the blank below
avg_num_trips_query = """
                      WITH trips_by_day AS
                      (
                      SELECT DATE(trip_start_timestamp) AS trip_date,
                          COUNT(*) as num_trips
                      FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                      WHERE trip_start_timestamp > '2016-01-01' AND trip_start_timestamp < '2016-04-01'
                      GROUP BY trip_date
                      ORDER BY trip_date
                      )
                      SELECT trip_date,
                          -- We want to find the Average of the trips
                          AVG(num_trips)
                          OVER (
                               -- We MUST order by date so the "rolling" window slides chronologically from day to day
                               ORDER BY trip_date
                               -- This defines our 7-day window. 
                               -- It looks at the 3 rows before the current row, the current row itself, 
                               -- and the 3 rows after the current row.
                               ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING
                               ) AS avg_num_trips
                      FROM trips_by_day
                      """

In [13]:
# Run avg_num_trips_query and display results as a dataframe
avg_num_trips_result = client.query(avg_num_trips_query).to_dataframe(create_bqstorage_client=False)
avg_num_trips_result

,trip_date,avg_num_trips
0,2016-03-25,85666.000000
1,2016-03-10,98769.714286
2,2016-01-28,80813.142857
3,2016-03-02,91927.000000
4,2016-02-26,92183.285714
...,...,...
86,2016-01-04,74260.571429
87,2016-02-06,81947.142857
88,2016-01-31,80539.000000
89,2016-03-23,88152.714286


In [14]:
# Amend the query below
trip_number_query = """
                    SELECT pickup_community_area,
                        trip_start_timestamp,
                        trip_end_timestamp,
                        -- RANK() is a window function that assigns a rank number to each row
                        -- PARTITION BY resets the rank counter for each unique community area,
                        -- so trips from area 1 and area 2 are numbered independently
                        -- ORDER BY trip_start_timestamp ranks trips in chronological order —
                        -- the earliest trip in a community area gets rank 1, the next gets rank 2, etc.
                        -- Ties (same timestamp, same area) receive the same rank, and the next rank is skipped
                        RANK() OVER (PARTITION BY pickup_community_area
                                     ORDER BY trip_start_timestamp) AS trip_number
                    FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                    WHERE DATE(trip_start_timestamp) = '2013-10-03'
                    """

In [15]:
# Run trip_number_query and display results as a dataframe
trip_number_result = client.query(trip_number_query).to_dataframe(create_bqstorage_client=False)
trip_number_result

,pickup_community_area,trip_start_timestamp,trip_end_timestamp,trip_number
0,4,2013-10-03 00:00:00+00:00,2013-10-03 00:15:00+00:00,1
1,4,2013-10-03 00:15:00+00:00,2013-10-03 00:15:00+00:00,2
2,4,2013-10-03 00:15:00+00:00,2013-10-03 00:15:00+00:00,2
3,4,2013-10-03 00:15:00+00:00,2013-10-03 00:30:00+00:00,2
4,4,2013-10-03 00:15:00+00:00,2013-10-03 00:30:00+00:00,2
...,...,...,...,...
84263,49,2013-10-03 07:45:00+00:00,2013-10-03 07:45:00+00:00,1
84264,49,2013-10-03 09:00:00+00:00,2013-10-03 09:00:00+00:00,2
84265,49,2013-10-03 09:45:00+00:00,2013-10-03 09:45:00+00:00,3
84266,49,2013-10-03 15:15:00+00:00,2013-10-03 16:30:00+00:00,4


In [16]:
# Fill in the blanks below
break_time_query = """
                   SELECT taxi_id,
                       trip_start_timestamp,
                       trip_end_timestamp,
                       -- ========================================================
                       -- HOW THIS WORKS:
                       -- TIMESTAMP_DIFF calculates the difference between two times.
                       -- We compare the CURRENT trip's start time, against the PREVIOUS trip's end time.
                       -- ========================================================
                       TIMESTAMP_DIFF(
                           trip_start_timestamp, 
                           
                           -- LAG() reaches "backwards" 1 row to get the previous trip's end time.
                           LAG(trip_end_timestamp) 
                               OVER (
                                    -- We partition by taxi_id so we don't accidentally 
                                    -- mix up Driver A's trips with Driver B's trips.
                                    PARTITION BY taxi_id 
                                    -- We order by start time so the trips are sorted chronologically.
                                    ORDER BY trip_start_timestamp), 
                                    
                           -- The output unit for TIMESTAMP_DIFF will be in MINUTES
                           MINUTE) as prev_break
                   FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                   WHERE DATE(trip_start_timestamp) = '2013-10-03' 
                   """

# Run the query, and return a pandas DataFrame (without BQ storage to avoid permission issues)
break_time_result = client.query(break_time_query).result().to_dataframe(create_bqstorage_client=False)
break_time_result.head()

,taxi_id,trip_start_timestamp,trip_end_timestamp,prev_break
0,178f2c246d4a291dda4c5f9ad179a67eb9a91b1f3673e1...,2013-10-03 13:30:00+00:00,2013-10-03 13:45:00+00:00,795
1,185565b2dfcacc95bacae0096afc6dd4bfa39d75a810e9...,2013-10-03 10:00:00+00:00,2013-10-03 10:00:00+00:00,330
2,185565b2dfcacc95bacae0096afc6dd4bfa39d75a810e9...,2013-10-03 12:45:00+00:00,2013-10-03 12:45:00+00:00,-30
3,2b4ac03cc3d5e9ff7901ee13f2b871aceb91df7b303a0f...,2013-10-03 18:15:00+00:00,2013-10-03 18:30:00+00:00,1065
4,2b4ac03cc3d5e9ff7901ee13f2b871aceb91df7b303a0f...,2013-10-03 19:15:00+00:00,2013-10-03 19:15:00+00:00,-30
